In [3]:
import pandas as pd

In [4]:
#1
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]
 
category_pool = {
    "billing": [
        {"question": "how do i get a refund for a failed transaction",
         "answer": "Refunds for failed transactions are credited back within 5-7 business days.",
         "keywords": "refund transaction failed"},
        {"question": "where can i download my invoice",
         "answer": "You can download invoices from the Billing History section of your account.",
         "keywords": "invoice download billing"},
    ],
    "account": [
        {"question": "how do i update my registered mobile number",
         "answer": "Go to Profile > Edit Details > Mobile Number to update it.",
         "keywords": "mobile number update"},
        {"question": "how do i delete my account permanently",
         "answer": "Contact support through the Help Center to request account deletion.",
         "keywords": "delete account permanently"},
    ],
    "general": [
        {"question": "do you have a mobile app",
         "answer": "Yes, our app is available on both Android and iOS.",
         "keywords": "mobile app android ios"},
        {"question": "how do i contact customer support",
         "answer": "You can reach support via in-app chat or email support@company.com.",
         "keywords": "contact support help"},
    ],
}
 
categories = ["billing", "account", "general"]

roll_number = "1024170208"
 
digits_str = [ch for ch in roll_number if ch.isdigit()]
last_two_digits = [int(d) for d in digits_str[-2:]]
print(f"\nLast two digits of roll number: {last_two_digits}")
 
personalized_entries = []
usage_count = {"billing": 0, "account": 0, "general": 0}
 
for d in last_two_digits:
    cat = categories[d % 3]
    variant_index = usage_count[cat] % len(category_pool[cat])
    template = category_pool[cat][variant_index]
    personalized_entries.append({
        "question": template["question"],
        "answer": template["answer"],
        "keywords": template["keywords"],
        "category": cat,
    })
    usage_count[cat] += 1
    print(f"digit {d} -> category[{d} % 3] = {cat}")
 
all_entries = fixed_entries + personalized_entries
df = pd.DataFrame(all_entries)
 
print("\nFinal 6-row FAQ DataFrame:")
print(df, "\n")


Last two digits of roll number: [0, 8]
digit 0 -> category[0 % 3] = billing
digit 8 -> category[8 % 3] = general

Final 6-row FAQ DataFrame:
                                         question  \
0                          what is the annual fee   
1                           how to reset password   
2                     what are your working hours   
3                           how can i pay the fee   
4  how do i get a refund for a failed transaction   
5                        do you have a mobile app   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  Refunds for failed transactions are credited b...   
5  Yes, our app is available on both Android and ...   

                    keywords category  
0      fee cost price charge  billing  
1       passw

In [5]:
#2
def score_entries(query, dataframe):

    query_words = set(query.lower().split())
    scored = dataframe.copy()
 
    def _score(row):
        entry_words = set((row["question"] + " " + row["keywords"]).lower().split())
        return len(query_words & entry_words)
 
    scored["score"] = scored.apply(_score, axis=1)
    matches = scored[scored["score"] > 0].sort_values("score", ascending=False)
    return matches[["question", "answer", "category", "score"]]

print("demo")
demo_query = "how do i pay the fee"
print(f"Query: '{demo_query}'")
print(score_entries(demo_query, df), "\n")

demo
Query: 'how do i pay the fee'
                                         question  \
3                           how can i pay the fee   
4  how do i get a refund for a failed transaction   
0                          what is the annual fee   
1                           how to reset password   
5                        do you have a mobile app   

                                              answer category  score  
3         You can pay via UPI, card, or net banking.  billing      5  
4  Refunds for failed transactions are credited b...  billing      3  
0                          The annual fee is Rs 500.  billing      2  
1                   Go to Settings > Reset Password.  account      1  
5  Yes, our app is available on both Android and ...  general      1   



In [6]:
#3
def same_category(category_name, dataframe):
    return dataframe[dataframe["category"] == category_name][["question", "answer", "keywords"]]

print("demo")
sample_category = personalized_entries[0]["category"]
print(f"Entries in category '{sample_category}':")
print(same_category(sample_category, df), "\n")

demo
Entries in category 'billing':
                                         question  \
0                          what is the annual fee   
3                           how can i pay the fee   
4  how do i get a refund for a failed transaction   

                                              answer  \
0                          The annual fee is Rs 500.   
3         You can pay via UPI, card, or net banking.   
4  Refunds for failed transactions are credited b...   

                    keywords  
0      fee cost price charge  
3        pay payment upi fee  
4  refund transaction failed   



In [7]:
#4
print(df[["question", "keywords"]], "\n")
 
entry_index = int(input("Enter the index (row number) of the entry to update: "))
new_keyword = input("Enter a new keyword to add: ").strip()
 
df.at[entry_index, "keywords"] = df.at[entry_index, "keywords"] + " " + new_keyword
print(f"\nUpdated keywords for row {entry_index}: {df.at[entry_index, 'keywords']}")
 
csv_filename = f"{roll_number}_faq_data.csv"
df.to_csv(csv_filename, index=False)
print(f"Saved updated DataFrame to {csv_filename}\n")

                                         question                   keywords
0                          what is the annual fee      fee cost price charge
1                           how to reset password       password reset login
2                     what are your working hours     hours timing open time
3                           how can i pay the fee        pay payment upi fee
4  how do i get a refund for a failed transaction  refund transaction failed
5                        do you have a mobile app     mobile app android ios 


Updated keywords for row 3: pay payment upi fee emi
Saved updated DataFrame to 1024170208_faq_data.csv



In [8]:
#5
category_counts = df.groupby("category").size()
print(category_counts, "\n")

category
account    1
billing    3
general    2
dtype: int64 



In [9]:
#6
def score_entries_with_ties(query, dataframe):

    query_words = set(query.lower().split())
    scored = dataframe.copy()
 
    def _score(row):
        entry_words = set((row["question"] + " " + row["keywords"]).lower().split())
        return len(query_words & entry_words)
 
    scored["score"] = scored.apply(_score, axis=1)
    matches = scored[scored["score"] > 0].sort_values("score", ascending=False)
 
    if matches.empty:
        print(f"No matches found for query: '{query}'")
        return matches
 
    top_score = matches["score"].max()
    tied = matches[matches["score"] == top_score]
 
    if len(tied) > 1:
        print(f"Query: '{query}' -> TIE at score {top_score} between {len(tied)} entries:")
    else:
        print(f"Query: '{query}' -> single best match (score {top_score}):")
    print(tied[["question", "answer", "category", "score"]], "\n")
 
    return matches
 
 
print("demo")
tie_query = "fee"
no_tie_query = "reset my password"
 
score_entries_with_ties(tie_query, df)
score_entries_with_ties(no_tie_query, df)

demo
Query: 'fee' -> TIE at score 1 between 2 entries:
                 question                                      answer  \
0  what is the annual fee                   The annual fee is Rs 500.   
3   how can i pay the fee  You can pay via UPI, card, or net banking.   

  category  score  
0  billing      1  
3  billing      1   

Query: 'reset my password' -> single best match (score 2):
                question                            answer category  score
1  how to reset password  Go to Settings > Reset Password.  account      2 



,question,answer,keywords,category,score
1,how to reset password,Go to Settings > Reset Password.,password reset login,account,2
